## 1. Preprocessing and retinal cell-state reconstruction

The public processed count matrix was loaded together with donor/disease metadata. Gene symbols were cleaned, counts were normalized for exploratory single-cell analysis, and nuclei were represented using PCA, a nearest-neighbor graph, UMAP, and Leiden clustering.

In [ ]:
adata = sc.AnnData(X=X_raw.copy())

adata.var_names = genes[1].str.split(" ").str[0].values
adata.var_names_make_unique()

adata.obs = metadata.copy()

print(adata)
print("\nConditions:")
print(adata.obs["condition_labels"].value_counts())

In [ ]:
adata = sc.AnnData(X=X_raw.copy())

# Clean human-readable gene symbols
adata.var_names = genes[1].str.split(" ").str[0].values
adata.var_names_make_unique()

# Attach donor and disease metadata
adata.obs = metadata.copy()

print(adata)
print("\nConditions:")
print(adata.obs["condition_labels"].value_counts())

In [ ]:
# Normalize for exploratory single-cell analysis
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Identify informative genes
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000
)

# Dimensionality reduction
sc.tl.pca(
    adata,
    n_comps=50,
    use_highly_variable=True
)

# Cell-cell similarity graph
sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30
)

# 2D visualization
sc.tl.umap(adata)

# Unsupervised clustering
sc.tl.leiden(
    adata,
    resolution=0.5
)

print("Number of Leiden clusters:", adata.obs["leiden"].nunique())

In [ ]:
muller_markers = [
    "RLBP1",
    "GLUL",
    "SLC1A3",
    "RGR"
]

sc.tl.score_genes(
    adata,
    gene_list=muller_markers,
    score_name="muller_score"
)

cluster_scores = (
    adata.obs
    .groupby("leiden", observed=True)["muller_score"]
    .mean()
    .sort_values(ascending=False)
)

display(cluster_scores.head(10))

best_cluster = cluster_scores.index[0]

print("\nHighest-scoring candidate Müller cluster:", best_cluster)
print(
    "Number of nuclei:",
    (adata.obs["leiden"] == best_cluster).sum()
)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "leiden",
        "RLBP1",
        "GLUL",
        "SLC1A3",
        "RGR"
    ],
    legend_loc="on data"
)

## 2. Identification of candidate Müller glia

A candidate Müller-glial population was identified by combining unsupervised Leiden clustering with expression of established Müller-associated markers including RLBP1, GLUL, SLC1A3 and RGR. Cluster identity was subsequently checked using cluster-specific differential expression rather than relying on a single marker.

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="leiden",
    groups=["4"],
    reference="rest",
    method="wilcoxon"
)

sc.pl.rank_genes_groups(
    adata,
    groups=["4"],
    n_genes=20
)

In [ ]:
sc.pl.umap(
    adata,
    color=["RLBP1", "GLUL", "SLC1A3", "RGR"],
    show=False
)

plt.savefig(
    "Muller_marker_UMAP.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 3. Donor-level pseudobulk aggregation

To avoid treating thousands of nuclei from the same retina as independent biological replicates, raw counts from candidate Müller nuclei were aggregated at the donor level. This produced one Müller-glial expression profile per retinal sample for downstream differential-expression analysis.

In [ ]:
muller_mask = adata.obs["leiden"] == "4"

muller_raw = sc.AnnData(
    X=X_raw[muller_mask.values].copy(),
    obs=adata.obs.loc[muller_mask].copy(),
    var=adata.var.copy()
)

print("Candidate Müller nuclei:", muller_raw.n_obs)
print()
print(muller_raw.obs["condition_labels"].value_counts())

In [ ]:
samples = muller_raw.obs["sample_labels"].unique()

pseudo_counts = []
pseudo_metadata = []

for sample in samples:

    mask = muller_raw.obs["sample_labels"] == sample

    summed = muller_raw.X[mask.values].sum(axis=0)
    summed = np.asarray(summed).reshape(1, -1)

    pseudo_counts.append(csr_matrix(summed))

    pseudo_metadata.append({
        "sample": sample,
        "condition": muller_raw.obs.loc[
            mask, "condition_labels"
        ].iloc[0],
        "n_muller_nuclei": int(mask.sum())
    })

pseudo_X = vstack(pseudo_counts, format="csr")
pseudo_meta = pd.DataFrame(pseudo_metadata)

print("Pseudobulk matrix:", pseudo_X.shape)

display(
    pseudo_meta.sort_values("n_muller_nuclei")
)

In [ ]:
keep_samples = pseudo_meta["n_muller_nuclei"] >= 50

pseudo_X_filt = pseudo_X[keep_samples.values].copy()

pseudo_meta_filt = (
    pseudo_meta.loc[keep_samples]
    .copy()
    .reset_index(drop=True)
)

print("Samples retained:", len(pseudo_meta_filt))
print()
print(pseudo_meta_filt["condition"].value_counts())

## 4. Donor-level differential expression

Raw Müller-glial counts were aggregated per retinal donor. Samples with fewer than 50 candidate Müller nuclei were excluded, leaving 15 donor-level pseudobulk profiles (5 healthy, 4 dry AMD, 6 wet AMD).

Differential-expression analysis was performed using donor-level raw counts rather than treating individual nuclei as independent biological replicates.

In [ ]:
counts_df = pd.DataFrame(
    pseudo_X_filt.toarray().astype(int),
    index=pseudo_meta_filt["sample"],
    columns=muller_raw.var_names
)

de_metadata = (
    pseudo_meta_filt
    .set_index("sample")[["condition"]]
)

# Remove genes with very little information
gene_keep = (counts_df >= 10).sum(axis=0) >= 3
counts_filtered = counts_df.loc[:, gene_keep]

print("Samples:", counts_filtered.shape[0])
print("Genes retained:", counts_filtered.shape[1])

display(de_metadata)

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

dds = DeseqDataSet(
    counts=counts_filtered,
    metadata=de_metadata,
    design="~condition",
    refit_cooks=True
)

dds.deseq2()

In [ ]:
stats_dry = DeseqStats(
    dds,
    contrast=["condition", "dAMD", "Healthy"]
)

stats_dry.summary()

dry_results = stats_dry.results_df.copy()
dry_results = dry_results.sort_values("padj")

print(
    "Dry AMD genes with FDR < 0.05:",
    (dry_results["padj"] < 0.05).sum()
)

display(
    dry_results[
        ["baseMean", "log2FoldChange", "stat", "pvalue", "padj"]
    ].head(15)
)

In [ ]:
stats_wet = DeseqStats(
    dds,
    contrast=["condition", "wAMD", "Healthy"]
)

stats_wet.summary()

wet_results = stats_wet.results_df.copy()
wet_results = wet_results.sort_values("padj")

print(
    "Wet AMD genes with FDR < 0.05:",
    (wet_results["padj"] < 0.05).sum()
)

display(
    wet_results[
        ["baseMean", "log2FoldChange", "stat", "pvalue", "padj"]
    ].head(15)
)

## 5. Hallmark gene-set enrichment analysis

Because coordinated biological programs can shift even when individual genes do not reach significance in a small donor cohort, all genes were ranked using the differential-expression test statistic and analysed using preranked Hallmark gene-set enrichment analysis (GSEA).

In [ ]:
import gseapy as gp

dry_rank = (
    dry_results[["stat"]]
    .dropna()
    .sort_values("stat", ascending=False)
)

wet_rank = (
    wet_results[["stat"]]
    .dropna()
    .sort_values("stat", ascending=False)
)

In [ ]:
dry_gsea = gp.prerank(
    rnk=dry_rank,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

dry_pathways = (
    dry_gsea.res2d
    .sort_values("FDR q-val")
    .reset_index(drop=True)
)

display(
    dry_pathways[
        ["Term", "NES", "NOM p-val", "FDR q-val"]
    ].head(15)
)

In [ ]:
wet_gsea = gp.prerank(
    rnk=wet_rank,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    permutation_num=1000,
    seed=42,
    verbose=False
)

wet_pathways = (
    wet_gsea.res2d
    .sort_values("FDR q-val")
    .reset_index(drop=True)
)

display(
    wet_pathways[
        ["Term", "NES", "NOM p-val", "FDR q-val"]
    ].head(15)
)

In [ ]:
sig_dry = dry_pathways[
    dry_pathways["FDR q-val"] < 0.05
].copy()

sig_wet = wet_pathways[
    wet_pathways["FDR q-val"] < 0.05
].copy()

print("Dry AMD significant pathways:", len(sig_dry))
print("Wet AMD significant pathways:", len(sig_wet))

In [ ]:
shared = sig_dry[
    ["Term", "NES", "FDR q-val"]
].merge(
    sig_wet[
        ["Term", "NES", "FDR q-val"]
    ],
    on="Term",
    suffixes=("_dry", "_wet")
)

shared = shared.sort_values("FDR q-val_dry")

display(shared)

## 6. Shared disease-associated transcriptional programs

Pathways significant at FDR < 0.05 in both dry and wet AMD were compared to identify transcriptional programs shared across the two disease phenotypes.

In [ ]:
plot_df = shared.sort_values("NES_dry", ascending=True).copy()

y = np.arange(len(plot_df))
height = 0.35

plt.figure(figsize=(9, 6))

plt.barh(
    y - height/2,
    plot_df["NES_dry"],
    height,
    label="Dry AMD"
)

plt.barh(
    y + height/2,
    plot_df["NES_wet"],
    height,
    label="Wet AMD"
)

plt.yticks(y, plot_df["Term"])
plt.xlabel("Normalized Enrichment Score (NES)")
plt.title(
    "Shared pathway enrichment in candidate Müller glia\n"
    "Dry and wet AMD vs healthy retina"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    "shared_AMD_Muller_pathways_final.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
dry_results.to_csv(
    "dry_AMD_vs_healthy_DE.csv"
)

wet_results.to_csv(
    "wet_AMD_vs_healthy_DE.csv"
)

dry_pathways.to_csv(
    "dry_AMD_GSEA.csv",
    index=False
)

wet_pathways.to_csv(
    "wet_AMD_GSEA.csv",
    index=False
)

shared.to_csv(
    "shared_significant_pathways.csv",
    index=False
)

pseudo_meta_filt.to_csv(
    "Muller_pseudobulk_sample_metadata.csv",
    index=False
)

print("All project outputs saved.")